# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[-0.1062,  0.2148,  0.1707,  0.1620, -0.0282, -0.0767, -0.1703, -0.1161],
        [ 0.1207, -0.3098, -0.1622, -0.0964, -0.0246,  0.2049,  0.0716,  0.0212]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cuda:0
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [6]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[-0.4812, -1.7955],
        [-0.0041,  0.1476],
        [ 0.1769, -0.3985],
        [ 0.2337, -0.1867],
        [-0.4281, -0.2426]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[ 0.1765, -0.3065],
        [-0.0828, -0.3015],
        [-0.0087, -0.2236],
        [-0.2630, -0.3990],
        [-0.1763, -0.3651]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [7]:
# import torch
# import torch.nn as nn

# class SimpleModel(nn.Module):
#     def __init__(self) -> None:
#         super().__init__()
#         self.fc = nn.Linear(4, 2)

#     # -----------------------------
#     # Using torch.no_grad() as method decorator
#     # -----------------------------
#     @torch.no_grad()
#     def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
#         return self.fc(x)

#     # -----------------------------
#     # Using torch.inference_mode() as method decorator
#     # -----------------------------
#     @torch.inference_mode()
#     def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
#         return self.fc(x)


# model = SimpleModel()
# x = torch.randn(5, 4)

# # -----------------------------
# # Call decorated methods
# # -----------------------------
# output_no_grad_method = model.forward_no_grad(x)
# output_infer_method = model.forward_inference(x)

# print("Output no_grad method:\n", output_no_grad_method)
# print("Output inference_mode method:\n", output_infer_method)

# # -----------------------------
# # Using context managers
# # -----------------------------

# with torch.no_grad():
#     output_no_grad_cm = model(x)

# with torch.inference_mode():
#     output_infer_cm = model(x)

# print("Output no_grad context manager:\n", output_no_grad_cm)
# print("Output inference_mode context manager:\n", output_infer_cm)


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [8]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc1.weight: requires_grad=False
fc1.bias: requires_grad=False
bn.weight: requires_grad=False
bn.bias: requires_grad=False
fc2.weight: requires_grad=False
fc2.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [9]:
# import torch
# import torch.nn as nn
# from torch import Tensor
# from typing import Self

# class CustomBlock(nn.Module):
#     def __init__(self) -> None:
#         super().__init__()
#         self.linear = nn.Linear(4, 4)
#         self.dropout = nn.Dropout(p=0.5)

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         x = self.linear(x)
#         x = torch.relu(x)
#         x = self.dropout(x)
#         return x

#     # -----------------------------
#     # Override train() method
#     # -----------------------------
#     def train(self, mode: bool = True) -> Self:
#         print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
#         super().train(mode)  # Call original method to update submodules
#         # Add any custom logic here
#         return self

#     # -----------------------------
#     # Override eval() method
#     # -----------------------------
#     def eval(self) -> Self:
#         print("CustomBlock set to eval mode")
#         return super().eval()


# # Example usage
# model = CustomBlock()
# x = torch.randn(2, 4)

# # Switch to training mode
# model.train()
# output_train = model(x)

# # Switch to evaluation mode
# model.eval()
# with torch.no_grad():
#     output_eval = model(x)

# print("Output training mode:", output_train)
# print("Output eval mode:", output_eval)


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [10]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [11]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [12]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [13]:
from torch import Tensor

def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [14]:
import torch
import torch.nn as nn
class LSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.gates = nn.Linear(input_size + hidden_size, 4*hidden_size)
        

        
    def forward(self, x, h_prev, c_prev):
        comb_input = torch.cat([x, h_prev], dim=-1)
        f, i, c_cand, o = self.gates(comb_input).chunk(4, dim=-1)
        f = torch.sigmoid(f)
        i = torch.sigmoid(i)
        c_cand = torch.tanh(c_cand)
        o = torch.sigmoid(o)
        
        c = f * c_prev + i * c_cand
        h = o * torch.tanh(c)
        
        return h, c

cell = LSTM(10, 16)
x = torch.randn(4, 10)
h0 = torch.zeros(4, 16)
c0 = torch.zeros(4, 16)   

h1, c1 = cell(x, h0, c0)
print(h1)

tensor([[ 0.0704,  0.0051, -0.0057,  0.0111, -0.0815, -0.0960,  0.0245,  0.0157,
          0.0365, -0.0019,  0.0147, -0.0273,  0.1143, -0.0172,  0.0609,  0.0870],
        [ 0.0734,  0.0496,  0.0539, -0.0309,  0.0169,  0.0228,  0.1011, -0.2147,
         -0.0471, -0.0276, -0.0692, -0.2064,  0.0768,  0.0237, -0.0477,  0.0618],
        [-0.0631,  0.0873,  0.0102, -0.1052,  0.0766,  0.1075,  0.0567,  0.0766,
         -0.2620,  0.0289, -0.2293, -0.0340,  0.1004, -0.1386, -0.2492, -0.0829],
        [ 0.1061,  0.0668,  0.0887,  0.0792, -0.0309,  0.0173,  0.0580, -0.1171,
         -0.0297,  0.0535, -0.1211, -0.1277,  0.1429, -0.1113, -0.0242, -0.0106]],
       grad_fn=<MulBackward0>)


### Inception

![inception](assets/inception.png)

In [15]:
class Inception_Block(nn.Module):
    def __init__(self, in_channels, ch_1x1, ch_3x3_reduce, ch_3x3, ch_5x5_reduce, ch_5x5, pool_proj):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, ch_1x1, kernel_size=1),
            nn.ReLU(inplace=True)
        )
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, ch_3x3_reduce, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_3x3_reduce,ch_3x3, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, ch_5x5_reduce, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch_5x5_reduce, ch_5x5, kernel_size=5, padding=2),
            nn.ReLU(inplace=True)
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)
    
i = Inception_Block(64, 16, 32, 32, 8, 16, 16)
x = torch.randn(2, 64, 28, 28)

print(i(x).shape)  
print(i(x))  

torch.Size([2, 80, 28, 28])
tensor([[[[0.5132, 0.0000, 0.2552,  ..., 0.2094, 0.0000, 0.5328],
          [0.7873, 0.2306, 0.1200,  ..., 0.1426, 0.0489, 0.3589],
          [0.3325, 0.0000, 0.0000,  ..., 1.0313, 0.0000, 0.0480],
          ...,
          [0.0000, 0.5556, 0.0000,  ..., 0.0000, 0.0000, 0.8921],
          [0.2376, 0.8985, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.2774, 0.0000, 0.7000,  ..., 0.0000, 0.0000, 1.2769]],

         [[0.8795, 0.5995, 0.2476,  ..., 0.0000, 0.5237, 1.8103],
          [0.0703, 0.2977, 0.3269,  ..., 0.0000, 0.4027, 0.2319],
          [0.2168, 0.3679, 0.5326,  ..., 0.6202, 0.0000, 0.0000],
          ...,
          [0.7214, 1.1398, 0.4727,  ..., 0.2814, 0.2220, 0.5294],
          [0.0000, 0.0000, 0.5667,  ..., 0.6161, 0.0000, 0.0000],
          [0.1369, 0.6379, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

         [[0.0494, 0.4395, 0.0000,  ..., 0.4573, 1.2884, 1.0498],
          [0.8481, 0.1106, 0.1873,  ..., 0.0082, 1.2139, 0.1131],
          [0.0

### SE

![se](assets/SqueezeAndExcite.png)

In [16]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels //reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels //reduction, channels),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1, 1)
        return x*s
    
m = SEBlock(64)
x = torch.randn(2, 64, 32, 32)
print(m(x).shape)  

torch.Size([2, 64, 32, 32])


### Selective Kernel

![selective](assets/SelectiveKernel.png)


In [17]:
class SelectiveKernel(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        
        self.conv3 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv5 = nn.Conv2d(channels, channels, kernel_size=5, padding=2)
        d = max(channels // reduction, 4)
        self.fc_gp = nn.Linear(channels, d)
        self.fc_a = nn.Linear(d, channels)
        self.fc_b = nn.Linear(d, channels)
        
    def forward(self, x):
        u1 = self.conv3(x)
        u2 = self.conv5(x)
        u = u1 + u2
        s = u.mean(dim=[2, 3])
        z = torch.relu(self.fc_gp(s))
        a = self.fc_a(z)
        b = self.fc_b(z)
        weights = torch.softmax(torch.stack([a, b], dim=1), dim=1)
        a_w = weights[:, 0].unsqueeze(-1).unsqueeze(-1)
        b_w = weights[:, 1].unsqueeze(-1).unsqueeze(-1)
        return u1*a_w + u2*b_w
        
        
m = SelectiveKernel(32)
x = torch.randn(2, 32, 16, 16)
print(m(x).shape)

torch.Size([2, 32, 16, 16])


### PatchMerger

![patchmerger](assets/PatchMerger.png)


In [18]:
import torch
import torch.nn as nn
class PatchMerger(nn.Module):
    def __init__(self, dim, num_output):
        super().__init__()
        
        self.norm = nn.LayerNorm(dim)
        self.weights = nn.Parameter(torch.randn(num_output, dim))
        
    def forward(self, x):
        x = self.norm(x)
        scores = torch.einsum('md,bnd->bmn', self.weights, x)
        attn = torch.softmax(scores, dim=-1)
        return torch.einsum('bmn,bnd->bmd', attn, x)
m = PatchMerger(64, 8)
x = torch.randn(2, 196, 64)
print(m(x).shape)

torch.Size([2, 8, 64])


## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [19]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + identity)
    
m = ResidualBlock(32)
x = torch.randn(2, 32, 16, 16)
print(m(x).shape)

torch.Size([2, 32, 16, 16])


## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [20]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        
        self.depthwise = nn.Conv2d(in_channels, in_channels,
                                   kernel_size, padding=padding, groups=in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)


    def forward(self, x):
        return self.pointwise(self.depthwise(x))
    
m = SeparableConv2d(16, 32)
x = torch.randn(2, 16, 20, 20)
print(m(x).shape)

torch.Size([2, 32, 20, 20])


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [21]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W_aligned = nn.Linear(d, d, bias=False)
        self.W_value = nn.Linear(d, d, bias = False)
        self.W_query = nn.Linear(d, d, bias = False)

    def forward(self, query, key):
        q_aligned = self.W_aligned(query).unsqueeze(1)
        score = torch.bmm(key, q_aligned.transpose(1,2)).squeeze(-1)
        att = torch.softmax(score, dim=1)
        context = torch.bmm(att.unsqueeze(1), key).squeeze(1)
        out = torch.tanh(self.W_value(context) + self.W_query(query))
        
        return out
m = VanillaAttention(d=32)

query = torch.randn(4, 32)
key = torch.randn(4, 10, 32)
print(m(query, key).shape)
        

torch.Size([4, 32])


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [22]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.d = d

    def forward(self, Q, K, V):
        S = torch.bmm(Q, K.transpose(1, 2)) / (self.d ** 0.5)
        
        attn = torch.softmax(S, dim=-1)
        return torch.bmm(attn, V)
m= ScaledDotProductAttention(16)

Q = torch.randn(2, 5, 16)
K = torch.randn(2, 7, 16)
V = torch.randn(2, 7, 16)
print(m(Q, K, V).shape)
print(m(Q, K, V))

torch.Size([2, 5, 16])
tensor([[[ 0.0705,  0.2634, -0.1870,  0.0294, -0.0783,  0.4714,  1.0656,
           0.5492,  0.0690, -0.1676,  0.4044, -0.7344, -0.1979, -0.9282,
           0.4590, -0.5966],
         [-0.6960,  0.0123, -1.5919, -0.1249,  0.1487,  0.6840, -0.0271,
          -0.1549, -0.3014,  0.5720, -0.1593, -0.1182, -0.3158,  0.2230,
          -0.4688,  0.7659],
         [-0.2454,  0.1100, -0.4491,  0.2049, -0.2170,  0.3959,  0.3006,
           0.3008, -0.1633, -0.0710,  0.0306, -0.1107,  0.2927, -0.3141,
          -0.3297,  0.1639],
         [-0.7156,  0.0433,  0.4491, -0.1430,  0.1383, -0.2038,  0.4122,
           0.4833,  0.1245, -0.8536,  0.6415, -0.7594,  0.1170,  0.0675,
           0.0141,  0.0562],
         [-0.0245,  0.0366,  0.1732,  0.0645, -0.2926,  0.2480,  0.0112,
           0.7410,  0.0630, -0.4632,  0.4026, -0.5824,  0.2909, -0.4421,
          -0.6234, -0.8552]],

        [[-0.3859,  0.2415, -0.4143, -1.2065, -0.8772, -0.2330, -0.0781,
           0.1052, -0.6818,

## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [23]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_head):
        super().__init__()
        self.heads = num_head
        self.d = d_model // num_head
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        
    def split_heads(self, x):
        b, l, d = x.shape
        return x.view(b, l, self.heads, self.d).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.split_heads(self.q_proj(query))
        K = self.split_heads(self.k_proj(key))
        V = self.split_heads(self.v_proj(value))
        
        
        S = torch.matmul(Q, K.transpose(-2, -1)) / (self.d ** 0.5)
        
        if mask is not None:
            S = S.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(S, dim=-1)
        out = torch.matmul(attn, V)
        out = out.transpose(1,2).contiguous().view(B, -1, self.heads * self.d)
        
        return self.out_proj(out)
    
m = MultiHeadAttention(32, 4)
x = torch.randn(2, 10, 32)
print(m(x,x,x).shape)

torch.Size([2, 10, 32])


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [24]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff),
                                 nn.GELU(),
                                 nn.Dropout(dropout),
                                 nn.Linear(d_ff, d_model),
                                 nn.Dropout(dropout)
                                 )

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        x = x + self.mlp(self.norm2(x))
        return x
m = TransformerEncoderLayer(32, 4, 64)
x = torch.randn(2, 10, 32)
print(m(x).shape)

torch.Size([2, 10, 32])


## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [25]:


class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches,dim, token_hidden, channel_hidden):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_patches, token_hidden),
            nn.GELU(),
            nn.Linear(token_hidden, num_patches)
        )
        
        self.norm2 = nn.LayerNorm(dim)
        self.channel_mlp = nn.Sequential(
            nn.Linear(dim, channel_hidden),
            nn.GELU(),
            nn.Linear(channel_hidden, dim)
        )

    def forward(self, x):
        y = self.norm1(x).transpose(1,2)
        y = self.token_mlp(y).transpose(1,2)
        x += y
        x += self.channel_mlp(self.norm2(x))
        return x
m = MLPMixerBlock(196, 64, 128, 256)
x = torch.randn(2, 196, 64)
print(m(x).shape)

torch.Size([2, 196, 64])


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [26]:
class ConvMixer(nn.Module):

    def __init__(self, channels, kernel_size=9):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size, padding=kernel_size //2,
                      groups = channels),
            nn.GELU(),
            nn.BatchNorm2d(channels)
        )
        
        self.pointwise = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=1),
            nn.GELU(),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x):
        x += self.depthwise(x)
        x = self.pointwise(x)
        return x

m = ConvMixer(32)
x = torch.randn(2, 32, 16, 16)
print(m(x).shape)

torch.Size([2, 32, 16, 16])


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ: ...

Все три механизма делают одну и туже вещь, они смешивают информацию по токенам и каналам.
По сути все они применяют две последовательные операции:

$Y = X + M_{token}(X)$

$Z = Y + M_{channel}(Y)$

Различается реализация $N_{tokens}$ токен миксинга.
Для Attention оно реализуется адаптивной матрицей, но он требует $N^2 C$

У MLP-Mixer receptove field глобальный без последовательного распространения информации через соседей.

У ConvMixer матрица взаимодействий сильно ограничена, но благодаря многократному применнию слоев, глобальную информацию все равно можно получить и стоит только $O(N)$


